# Match The Echo-Shifted Scans for CPR

In [5]:
import src.read_write as rw
import src.reconstruction as recon
from pathlib import Path
import numpy as np
import time


# ---- Directories ----
dir_h5 = Path('data/raw')

dir_reconstruction = Path('data/all_reconstructions')
dir_reconstruction.mkdir(parents=True, exist_ok=True)

dir_b0_maps = Path('data/all_b0_maps')
dir_b0_maps.mkdir(parents=True, exist_ok=True)

dir_run_info = Path('data/all_run_info')
dir_run_info.mkdir(parents=True, exist_ok=True)

dir_idc = Path('data/image_domain_correction')
dir_idc.mkdir(parents=True, exist_ok=True)

dir_idc_results = Path('data/image_domain_correction/results')
dir_idc_results.mkdir(parents=True, exist_ok=True)



list_ES200 = []
list_ES000 = []

for file in dir_h5.iterdir():

        if "ES200" in file.name:
            list_ES200.append(file)

for file in list_ES200:
    ES000_file = file.with_name(file.name.replace("ES200", "ES000"))
    if ES000_file.exists():
        list_ES000.append(ES000_file)

print(list_ES200)
print(list_ES000)


[PosixPath('data/raw/position1_ROreg_ES200.h5'), PosixPath('data/raw/position3_ROreg_ES200.h5'), PosixPath('data/raw/position2_ROreg_ES200.h5')]
[PosixPath('data/raw/position1_ROreg_ES000.h5'), PosixPath('data/raw/position3_ROreg_ES000.h5'), PosixPath('data/raw/position2_ROreg_ES000.h5')]


# Run Iterative CPR and Export Reconstructions and B0 maps

In [6]:


runtimes_all = []

list_ssh_orders = [2, 4, 8, 12]


for i in range(len(list_ES200)):

    runtimes_position = []

    position = list_ES000[i].stem.split("_")[0]

    # ---- Get Necessary Parameters ----
    xml000, kspace000 = rw.load_h5(list_ES000[i])
    xml200, kspace200 = rw.load_h5(list_ES200[i])


    dwellTime000 = rw.GetParam_from_xml("dwell", xml000, print_all = False)[0]*1e-03
    dwellTime200 = rw.GetParam_from_xml("dwell", xml200, print_all = False)[0]*1e-03


    fov000 = rw.GetParam_from_xml("fov", xml000, print_all = False)*1e-03
    fov200 = rw.GetParam_from_xml("fov", xml200, print_all = False)*1e-03


    te000 = rw.GetParam_from_xml("TE", xml000, print_all = False)[0]
    te200 = rw.GetParam_from_xml("TE", xml200, print_all = False)[0]

    t_shift = abs(te000 - te200)/1000

    for ssh_order in list_ssh_orders:

        print(f"\n \n \nStarting CPR at {position} with SSH order {ssh_order}...\n")

        start = time.perf_counter()

        final_img_1, final_img_2, b0, history = recon.iterative_cpr(
                                                    kspace_regul = kspace000,
                                                    kspace_shift = kspace200,
                                                    fov = fov000,
                                                    t_shift = t_shift,
                                                    dwell_time = dwellTime000,
                                                    n_iter=5,
                                                    max_order=ssh_order,
                                                    sigma=1.5,
                                                    readout_axis=0,
                                                )

        runtime = time.perf_counter() - start

        runtimes_position.append(runtime)

        # ---- EXPORT Last CPR and b0 estimation ----
        name_000 = list_ES000[i].stem
        name_200 = list_ES200[i].stem
        position = name_000.split("_")[0]



        rw.write_nii_from_xml(abs(final_img_1), xml000,   f'{dir_reconstruction}/{name_000}_CPRIter_Order{ssh_order:02d}.nii.gz')
        #rw.write_nii_from_xml(abs(final_img_2), xml200,   f'{dir_reconstruction}/{name_200}_CPRIter_Order{ssh_order:02d}.nii.gz')

        np.save(f'{dir_b0_maps}/b0_map_{position}_CPRIter_Order{ssh_order:02d}.npy', b0)

        # ---- EXPORT First CPR and b0 estimation (No iterative CPR) ----
        cpr_one_000 = history["u1"][1]
        cpr_one_200 = history["u2"][1]
        b0_one = history["b0"][0]


        rw.write_nii_from_xml(abs(cpr_one_000), xml000,   f'{dir_reconstruction}/{name_000}_CPROne_Order{ssh_order:02d}.nii.gz')
        #rw.write_nii_from_xml(abs(cpr_one_200), xml200,   f'{dir_reconstruction}/{name_200}_CPROne_Order{ssh_order:02d}.nii.gz')

        np.save(f'{dir_b0_maps}/b0_map_{position}_CPROne_Order{ssh_order:02d}.npy', b0_one)

    runtimes_all.append(runtimes_position)



 
 
Starting CPR at position1 with SSH order 2...

Iteration 1/5
Iteration 2/5
Iteration 3/5
B0 changes: d(k)= 2.370e-01, d(k-1)= 6.284e-01
Iteration 4/5
B0 changes: d(k)= 1.162e-01, d(k-1)= 2.370e-01
Iteration 5/5
B0 changes: d(k)= 6.565e-02, d(k-1)= 1.162e-01
img shape :  (111, 111, 111)
Saved NIfTI file to data/all_reconstructions/position1_ROreg_ES000_CPRIter_Order02.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/all_reconstructions/position1_ROreg_ES000_CPROne_Order02.nii.gz

 
 
Starting CPR at position1 with SSH order 4...

Iteration 1/5
Iteration 2/5
Iteration 3/5
B0 changes: d(k)= 2.263e-01, d(k-1)= 5.932e-01
Iteration 4/5
B0 changes: d(k)= 1.149e-01, d(k-1)= 2.263e-01
Iteration 5/5
B0 changes: d(k)= 6.181e-02, d(k-1)= 1.149e-01
img shape :  (111, 111, 111)
Saved NIfTI file to data/all_reconstructions/position1_ROreg_ES000_CPRIter_Order04.nii.gz
img shape :  (111, 111, 111)
Saved NIfTI file to data/all_reconstructions/position1_ROreg_ES000_CPROne_Order04.nii.gz


# Runtime
runtimes_all = [
    
    [time_ssh2, time_ssh4, time_ssh8, time_ssh12],  # position 0
    [time_ssh2, time_ssh4, time_ssh8, time_ssh12],  # position 1
    [time_ssh2, time_ssh4, time_ssh8, time_ssh12]   # position 2
]

In [8]:


print(runtimes_all[0])
print(runtimes_all[1])
print(runtimes_all[2])
np.save(f'{dir_run_info}/runtimes_CPRIter.npy', np.array(runtimes_all))

[45.444124582980294, 43.580338917003246, 47.71602374999202, 52.27130158399814]
[43.36258991601062, 44.39168349999818, 47.16245441601495, 51.8011331250018]
[41.73642512501101, 43.70913495900459, 48.951918874983676, 52.71114062500419]
